# SAPC Unmatched Pharmacy Geocoder

Geocodes SAPC pharmacies that had no confident match to the existing geocoded list (`sapc_needs_geocoding.csv`).

Uses the same Google Places Text Search pattern as the main extraction notebook. The output schema matches `PHARMACIES_places_full_results.csv` so both files can be stacked for the master build.

Outputs: `SAPC_places_results.csv`

## Imports and configuration

In [ ]:
import os, time
import pandas as pd
import requests

SAPC_UNMATCHED_PATH = r'sapc_needs_geocoding.csv'
OUTPUT_PATH     = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\SAPC_places_results.csv'
CHECKPOINT_PATH = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\SAPC_places_results_checkpoint.csv'

API_KEY = os.getenv('GOOGLE_MAPS_API_KEY')
if not API_KEY:
    raise ValueError('GOOGLE_MAPS_API_KEY not found in environment variables.')

SLEEP_SECONDS = 0.2
SAVE_EVERY    = 100

print('Config ready.')

## Load and inspect

In [ ]:
sapc = pd.read_csv(SAPC_UNMATCHED_PATH, dtype=str)
print(f'SAPC unmatched: {len(sapc):,} rows')
print(f'Columns: {list(sapc.columns)}')
print()
print('Status breakdown:')
print(sapc['sapc_status'].value_counts(dropna=False).to_string())
print()
print('Province breakdown:')
print(sapc['sapc_province'].value_counts(dropna=False).to_string())
display(sapc.head(3))

## Helper functions

`build_query` is adapted for SAPC column names. The `search_place_text` and `needs_review` functions are identical to those in the main extraction notebook.

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    return x if x else None


def build_query(row):
    # Format: 'Pharmacy Name, Street Address, City, Province, South Africa'
    parts = [p for p in [
        clean_text(row.get('sapc_name')),
        clean_text(row.get('sapc_address')),
        clean_text(row.get('sapc_city')),
        clean_text(row.get('sapc_province')),
        'South Africa',
    ] if p]
    return ', '.join(parts)


def search_place_text(query, api_key):
    url    = 'https://maps.googleapis.com/maps/api/place/textsearch/json'
    params = {'query': query, 'type': 'pharmacy', 'key': api_key}
    r      = requests.get(url, params=params, timeout=30)
    try:
        data = r.json()
    except Exception:
        return {
            'http_status': r.status_code, 'api_status': 'BAD_JSON',
            'place_id': None, 'matched_name': None, 'matched_address': None,
            'lat': None, 'lng': None, 'types': None, 'raw_error': r.text[:500],
        }
    status = data.get('status')
    if status != 'OK':
        return {
            'http_status': r.status_code, 'api_status': status,
            'place_id': None, 'matched_name': None, 'matched_address': None,
            'lat': None, 'lng': None, 'types': None, 'raw_error': str(data)[:500],
        }
    result = data['results'][0]
    return {
        'http_status':     r.status_code,
        'api_status':      status,
        'place_id':        result.get('place_id'),
        'matched_name':    result.get('name'),
        'matched_address': result.get('formatted_address'),
        'lat':             result.get('geometry', {}).get('location', {}).get('lat'),
        'lng':             result.get('geometry', {}).get('location', {}).get('lng'),
        'types':           ', '.join(result.get('types', [])) if result.get('types') else None,
        'raw_error':       None,
    }


def needs_review(row):
    if row['api_status'] != 'OK': return True
    if pd.isna(row['place_id']): return True
    if pd.isna(row['lat']): return True
    if pd.isna(row['lng']): return True
    return False


print('Helpers defined.')

## Build query strings and resume from checkpoint

Resumes from an existing checkpoint if one is present. Re-running this cell after an interruption will skip already-processed records.

In [ ]:
sapc['query_string'] = sapc.apply(build_query, axis=1)
sapc = sapc[sapc['query_string'].notna() & (sapc['query_string'].str.len() > 0)].copy()
print(f'Records with usable query string: {len(sapc):,}')

if os.path.exists(CHECKPOINT_PATH):
    existing     = pd.read_csv(CHECKPOINT_PATH, dtype=str)
    done_queries = set(existing['query_string'].dropna().tolist())
    remaining    = sapc[~sapc['query_string'].isin(done_queries)].copy()
    print(f'Checkpoint found: {len(existing):,} already done')
    print(f'Remaining to query: {len(remaining):,}')
else:
    existing  = pd.DataFrame()
    remaining = sapc.copy()
    print(f'No checkpoint — querying all {len(remaining):,} records')

## Run geocoding

Checkpoints every 100 records. The output schema matches `PHARMACIES_places_full_results.csv`.

In [ ]:
results = []

for n, (_, row) in enumerate(remaining.iterrows(), start=1):
    query  = row['query_string']
    result = search_place_text(query, API_KEY)

    results.append({
        'sapc_y_number':          row.get('sapc_y_number'),
        'sapc_status':            row.get('sapc_status'),
        'sapc_owner':             row.get('sapc_owner'),
        'sapc_responsible_pharm': row.get('sapc_responsible_pharm'),
        'sapc_licence':           row.get('sapc_licence'),
        'sapc_registration_date': row.get('sapc_registration_date'),
        'sapc_inspection':        row.get('sapc_inspection'),
        'practice_name':          row.get('sapc_name'),
        'address':                row.get('sapc_address'),
        'city':                   row.get('sapc_city'),
        'province':               row.get('sapc_province'),
        'query_string':           query,
        **result,
    })

    if n % 100 == 0:
        print(f'Processed {n:,} / {len(remaining):,}')

    if n % SAVE_EVERY == 0:
        chk_new = pd.DataFrame(results)
        if len(chk_new) > 0:
            chk_new['needs_review'] = chk_new.apply(needs_review, axis=1)
        chk_combined = pd.concat([existing, chk_new], ignore_index=True)
        chk_combined.drop_duplicates(subset=['query_string'], keep='first', inplace=True)
        chk_combined.to_csv(CHECKPOINT_PATH, index=False)
        print(f'  Checkpoint saved ({len(chk_combined):,} total)')

    time.sleep(SLEEP_SECONDS)

print(f'\nDone. {len(results):,} new records queried.')

## Combine, flag and save

In [ ]:
new_results = pd.DataFrame(results)
if len(new_results) > 0:
    new_results['needs_review'] = new_results.apply(needs_review, axis=1)

combined = pd.concat([existing, new_results], ignore_index=True)
combined.drop_duplicates(subset=['query_string'], keep='first', inplace=True)

if 'needs_review' in combined.columns:
    combined['needs_review'] = combined['needs_review'].apply(
        lambda x: True if str(x).strip().lower() in ('true', '1') else False
    )

combined.to_csv(OUTPUT_PATH, index=False)
print(f'Saved {len(combined):,} records -> {OUTPUT_PATH}')
print()
print('API status breakdown:')
print(combined['api_status'].value_counts(dropna=False).to_string())
print()
print('Needs review breakdown:')
print(combined['needs_review'].value_counts(dropna=False).to_string())

geocoded = combined[combined['place_id'].notna()]
print(f'\nSuccessfully geocoded: {len(geocoded):,} / {len(combined):,}')
display(geocoded.head(3))

## Append to master geocoded file

Merges successfully geocoded SAPC records into the master geocoded file. Run this only after reviewing the results above.

In [ ]:
master = pd.read_csv('geocoded_with_sapc.csv', dtype=str)
print(f'Master file: {len(master):,} rows')

to_append = combined[
    combined['place_id'].notna() &
    (combined['needs_review'] == False)
].copy()
print(f'Records to append: {len(to_append):,}')

master_updated = pd.concat([master, to_append], ignore_index=True)
master_updated.drop_duplicates(subset=['place_id'], keep='first', inplace=True)

MASTER_UPDATED_PATH = r'C:\Users\jcahi\Box\DAIR_SA\Data Sets\Pharmacy Data\Cleaned Datasets\geocoded_with_sapc_complete.csv'
master_updated.to_csv(MASTER_UPDATED_PATH, index=False)

print(f'\nComplete master: {len(master_updated):,} rows -> geocoded_with_sapc_complete.csv')
print(f'  Original master : {len(master):,}')
print(f'  Newly added     : {len(master_updated) - len(master):,}')